In [2]:
! pip install rootutils

In [8]:
%pip install opencv-python

  Using cached opencv_python-4.12.0.88-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
Using cached opencv_python-4.12.0.88-cp37-abi3-macosx_13_0_arm64.whl (37.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 3.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [opencv-python]0m [opencv-python]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 6.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
%pip install rootutils

  Using cached rootutils-1.0.7-py3-none-any.whl.metadata (4.7 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached rootutils-1.0.7-py3-none-any.whl (6.4 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [rootutils]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
%pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 3.9 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import cv2
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import rootutils
from scipy.spatial.transform import Rotation

#  auto-reloads all modules
%load_ext autoreload
%autoreload 2 

# import python modules relatively to the project root directory
root = rootutils.setup_root(".", indicator="homeworks", pythonpath=True)

# data directory
DATA_DIR = root / "homeworks/data"
print(f"DATA_DIR: {DATA_DIR}")

DATA_DIR: /Users/lily/Desktop/HSE/image-processing/homeworks/data


In [6]:
# вспомогательные функции для визуализации

# функция для визуализации системы координат
def plot_coordinate_system(
    R=np.eye(3), t=np.array([0, 0, 0]), labels=("u", "v", "w"), length=1.0, plots=None
):
    if plots is None:
        plots = []

    origin = t.flatten()

    u_axis = R[:, 0] * length + origin
    v_axis = R[:, 1] * length + origin
    w_axis = R[:, 2] * length + origin

    p = px.line_3d(
        x=[origin[0], u_axis[0]],
        y=[origin[1], u_axis[1]],
        z=[origin[2], u_axis[2]],
    )
    p.update_traces(
        line=dict(color="red", width=5),
        mode="lines+text",
        text=["", labels[0]],
    )
    plots.append(p)

    p = px.line_3d(
        x=[origin[0], v_axis[0]], y=[origin[1], v_axis[1]], z=[origin[2], v_axis[2]]
    )
    p.update_traces(
        line=dict(color="green", width=5),
        mode="lines+text",
        text=["", labels[1]],
    )
    plots.append(p)

    p = px.line_3d(
        x=[origin[0], w_axis[0]], y=[origin[1], w_axis[1]], z=[origin[2], w_axis[2]]
    )
    p.update_traces(
        line=dict(color="blue", width=5),
        mode="lines+text",
        text=["", labels[2]],
    )
    plots.append(p)

    return plots


# функция для визуализации точек и рёбер в 3D
def plot_points(points, edges, plots=None):
    if plots is None:
        plots = []

    u = points[:, 0]
    v = points[:, 1]
    w = points[:, 2]

    num_points = len(points)

    p = px.scatter_3d(
        x=u,
        y=v,
        z=w,
        color=num_points * ["rgb(255, 0, 255)"],
        color_discrete_map="identity",
    )
    p.update_traces(marker_size=2)
    plots.append(p)

    if edges is None:
        return plots

    for n, m in edges:
        u1, v1, w1 = points[n]
        u2, v2, w2 = points[m]

        p = px.line_3d(x=[u1, u2], y=[v1, v2], z=[w1, w2])
        p.update_traces(line=dict(color="gray", width=3))
        plots.append(p)

    return plots


# функция для отображения 3D сцены
def show_plots(plots, width, height, eye):
    data = []

    for p in plots:
        data += p.data

    fig = go.Figure(data)

    fig.update_layout(
        autosize=True,
        width=width,
        height=height,
        paper_bgcolor="white",
        scene=dict(
            aspectmode="data",
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
        ),
    )

    view = dict(
        eye=dict(x=eye[0], y=eye[1], z=eye[2]),
        center=dict(x=0, y=0, z=0),
        up=dict(x=0, y=0, z=1),
    )

    fig.update_layout(scene_camera=view)
    fig.show()

#### **Преобразование системы координат**

$$
    \mathbf{w}_1 = \mathbf{R}\cdot\mathbf{w}_2 + \mathbf{t}
$$
где:
- $ \mathbf{R} $ — матрица вращения (3x3)
- $ \mathbf{t} $ — вектор сдвига (3x1)

In [7]:
# загрузка 3D сцены
scene = np.load(DATA_DIR / "box.npz")

# 3D точки и рёбра сцены
points1 = scene["vertices"]  # (u1, v1, w1) координаты точек в 3D
edges = scene["edges"]  # рёбра сцены (индексы точек)

print(f"Points shape:\n {points1.shape}")
print(f"Edges:\n {edges}")

Points shape:
 (160, 3)
Edges:
 [[ 20  99]
 [  0 119]]


In [21]:
%pip install pandas

  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 6.0 MB/s eta 0:00:0000:010:01
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
%pip install "nbformat>=4.2.0"


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
# визуализация 3D точек и рёбер
plots = plot_points(points1, edges)

# визуализация системы координат
plots = plot_coordinate_system(
    labels=("u1", "v1", "w1"),
    plots=plots,  # добавление к существующим графикам
)

# вывод графика
show_plots(
    width=600,
    height=600,  # размеры окна
    eye=(2, 2, 2),  # координаты точки обзора сцены
    plots=plots,
)

In [9]:
# рассмотрим другую систему координат (u2, v2, w2)
# переход из системы координат (u2, v2, w2)
# в систему координат (u1, v1, w1)
# задается поворотом и сдвигом

R = np.array([[0, 0, -1], [1, 0, 0], [0, -1, 0]])

t = np.array([6, 0, 5])

# визуализация системы координат (u2, v2, w2)
plots = plot_coordinate_system(R=R, t=t, labels=("u2", "v2", "w2"), plots=plots)

# вывод графика
show_plots(
    width=600,
    height=600,  # размеры окна
    eye=(2, 2, 2),  # координаты точки обзора сцены
    plots=plots,
)

#### **Задача 1**

Реализуйте функцию `transform_points`, которая вычисляет координаты точек `points1` в новой системе координат `(u2, v2, w2)`, заданной поворотом `R` и сдвигом `t`.

```python
# file: homeworks/homework_07.py

def transform_points(points1, R, t):
    ...
    return points2
```

In [10]:
points1.shape

(160, 3)

Проверка работоспособности функции `transform_points`:

In [22]:
def transform_points(points1, R, t):
    points2 = (R.T @ (points1 - t).T).T
    return points2


points2 = transform_points(points1, R, t)

# визуализация 3D точек и рёбер в системе координат (u2, v2, w2)
plots = plot_points(points2, edges)

# визуализация системы координат (u2, v2, w2)
plots = plot_coordinate_system(labels=("u2", "v2", "w2"), plots=plots)

# вывод графика
show_plots(
    width=600,
    height=600,  # размеры окна
    eye=(2, 2, 2),  # координаты точки обзора сцены
    plots=plots,
)

#### **Параметризация матрицы вращения с помощью вектора вращения.  Формулы Родрига**

- Вектор вращения:
    $$
        \boldsymbol{\omega} = \theta\cdot\mathbf{n}
    $$

- Матрица вращения:
    $$
        \mathbf{R} = \mathbf{I} + \sin\theta\,[\mathbf{n}]_\times + (1-\cos\theta)\,[\mathbf{n}]_\times^2
    $$


    

#### **Задача 2**

- Реализуйте функцию `rotation_matrix_from_rotvec(omega)`, которая по вектору вращения `omega` вычисляет матрицу поворота `R` с помощью формулы Родрига.



- Реализуйте функцию `rotvec_from_rotation_matrix(R)`, которая по матрице поворота `R` вычисляет вектор вращения `omega` с помощью обратной формулы Родрига.

```python
    # file: homeworks/homework_07.py

    def rotation_matrix_from_rotvec(omega):
        ...
        return R
    
    def rotvec_from_rotation_matrix(R):
        ...
        return omega
```

Тестирование функций `rotation_matrix_from_rotvec` и `rotvec_from_rotation_matrix`:

In [138]:
from homework_07 import rotation_matrix_from_rotvec


def rotvec_from_rotation_matrix(R):
    theta = np.arccos((np.trace(R) - 1) / 2)
    omega = (theta / (2 * np.sin(theta))) * np.array(
        [R[2][1] - R[1][2], R[0][2] - R[2][0], R[1][0] - R[0][1]]
    )
    return omega


# вектор вращения вокруг оси (1, 1, 1) на угол 45 градусов
omega = np.array([1, 1, 1])
omega = omega / np.linalg.norm(omega)
omega = omega * (np.pi / 4)

# вычисление матрицы поворота из вектора вращения
R = rotation_matrix_from_rotvec(omega)

# проверка результата с помощью scipy
scipy_R = Rotation.from_rotvec(omega).as_matrix()
assert np.allclose(R, scipy_R)

# вычисление вектора вращения из матрицы поворота
recovered_omega = rotvec_from_rotation_matrix(R)

# проверка результата
assert np.allclose(omega, recovered_omega)

#### **Формирование изображения**

Считаем, что выполнена калибровка камеры и известны ее внутренние параметры
$$
    \mathbf{K} = \begin{bmatrix}
    f_x & s & c_x \\
    0 & f_y & c_y \\
    0 & 0 & 1
    \end{bmatrix}
$$
и внешние параметры - матрица вращения и вектор сдвига из мировой системы координат в систему координат камеры
$$
    \mathbf{R}\qquad \mathbf{t}
$$

In [66]:
# внутренние параметры камеры

# фокусные расстояния в пикселях
fx = fy = 500

# координаты principal point в пикселях
cx, cy = 320, 240

# матрица внутренних параметров камеры
K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

In [67]:
# внешние параметры камеры

# вектор вращения
omega = np.array([0.26029898, 0.26029898, 0.03426901])

# матрица вращения
R = rotation_matrix_from_rotvec(omega)

# вектор сдвига
t = np.array([-2.0, 2.0, 0.0])

In [68]:
# отобразим 3D точки в мировой системе координат
# и систему координат камеры

plots = plot_points(points1, edges)  # 3D точки

plots = plot_coordinate_system(  # мировая система координат
    labels=("u", "v", "w"), plots=plots
)

plots = plot_coordinate_system(  # система координат камеры
    R=R, t=t, labels=("uc", "vc", "wc"), plots=plots
)

# построение сцены
show_plots(
    width=600,
    height=600,  # размеры окна
    eye=(1.5, 1.5, 1.5),  # координаты точки обзора сцены
    plots=plots,
)

Вычислим матрицу камеры
$$
    \mathbf{P} = \mathbf{K} \begin{bmatrix}
    \mathbf{R} & \mathbf{t}
    \end{bmatrix}
$$

In [69]:
# G = [R  t]
G = np.zeros((3, 4))
G[:3, :3] = R
G[:3, 3] = t

# вычислим матрицу камеры
P = K @ G

with np.printoptions(precision=2, suppress=True):
    print(f"Camera matrix P:\n {P}")

Camera matrix P:
 [[  402.96    82.82   427.97 -1000.  ]
 [  -26.51   545.08    98.92  1000.  ]
 [   -0.25     0.26     0.93     0.  ]]


####  **Задача 3**

Реализуйте функцию `project_points(points, P)`, которая проецирует 3D точки в систему координат изображения с помощью матрицы камеры `P`.

$$
    \lambda\cdot\widetilde{\mathbf{x}}
    =
    \mathbf{P}\cdot
    \widetilde{\mathbf{w}}
$$

```python
    # file: homeworks/homework_07.py
    
    def project_points(points3D, P):
        ...
        return points2D
```

Проверка функции `project_points`:

In [77]:
def project_points(points3D, P):
    n = len(points3D)
    points2D = np.hstack((points3D, np.ones((n, 1)))) @ P.T
    lambd = points2D[:, 2].reshape(-1, 1)
    points2 = points2D[:, :2] / lambd
    return points2


points3D = points1

# проекция 3D точек в систему координат изображения
points2D = project_points(points3D, P)

# проверка результата с помощью OpenCV
cv_points2D, _ = cv2.projectPoints(
    points3D[None],
    omega.reshape(3, 1),
    t.reshape(3, 1),
    K,
    np.zeros((5, 1)),  # коэффициеты дисторсии
)

assert np.allclose(points2D, cv_points2D.reshape(-1, 2))

# визуализация cпроецированных точек на изображении
image_width = int(2 * cx)
image_height = int(2 * cy)

image = np.zeros((image_height, image_width, 3), dtype=np.uint8)

for x, y in points2D.astype(int):
    cv2.circle(image, (x, y), 3, (255, 0, 255), -1)

fig = px.imshow(image, height=400)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.show()

#### **Измерение расстояния между объектами**

In [79]:
src_image = cv2.imread(DATA_DIR / "station.png", cv2.IMREAD_COLOR_RGB)

fig = px.imshow(src_image, width=700)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.show()

На этой сцене:

- начало мировой системы координат находиться на жд. платформе

- ось $u$ направлена направо, перпендикулярно рельсам

- ось $v$ направлена вертикально вниз

- ось $w$ направлена вдоль рельс

Зададим внутренние и внешние параметры камеры:

In [80]:
# Внутренние параметры камеры
f, cx, cy = 810.5, 480, 270
K = np.array([[f, 0, cx], [0, f, cy], [0, 0, 1]])

# Внешние параметры камеры
omega = np.array([0.32828207, 0.13615168, -0.05789331])
R = Rotation.from_rotvec(omega).as_matrix()
t = np.array([0.26068896, 3.12807364, 1.05037924])

Построим матрицу камеры 
$$
    \mathbf{P} = \mathbf{K} \begin{bmatrix}
    \mathbf{R} & \mathbf{t}
    \end{bmatrix}
$$
и найдем оптический центр камеры в мировой системе координат:
$$
    \mathbf{0} = \mathbf{R}\cdot\mathbf{c} + \mathbf{t} 
    \qquad \Rightarrow \qquad
    \mathbf{c} = -\mathbf{R}^\top\cdot\mathbf{t}
$$

In [82]:
# G = [R  t]
G = np.zeros((3, 4))
G[:3, :3] = R
G[:3, 3] = t

# вычислим матрицу камеры
P = K @ G

# оптический центр камеры в мировых координатах
c = -R.T @ t

На трехмерной сцене, в мировой системе координат, 
проведем координатную сетку:
$$
    u = -2,-1, 0, 1, 2\qquad
    w = 5, 6, \ldots, 35\qquad
    v = 0
$$
и спроецируем ее на изображение:

In [83]:
grid_u = (-2, 3)  # в метрах
grid_w = (5, 36)  # в метрах

image1 = src_image.copy()

for u in range(*grid_u):
    s = [u, 0, grid_w[0]]  # начало отрезка
    e = [u, 0, grid_w[1] - 1]  # конец отрезка

    # проекция начала и конца отрезка на изображение
    p = K @ (R @ s + t)
    q = K @ (R @ e + t)

    # рисование отрезка на изображении
    cv2.line(
        image1,
        (int(p[0] / p[2]), int(p[1] / p[2])),
        (int(q[0] / q[2]), int(q[1] / q[2])),
        (255, 0, 255),
        1,
    )

for z in range(*grid_w):
    s = [grid_u[0], 0, z]  # начало отрезка
    e = [grid_u[1] - 1, 0, z]  # конец отрезка

    # проекция начала и конца отрезка на изображение
    p = K @ (R @ s + t)
    q = K @ (R @ e + t)

    # рисование отрезка на изображении
    cv2.line(
        image1,
        (int(p[0] / p[2]), int(p[1] / p[2])),
        (int(q[0] / q[2]), int(q[1] / q[2])),
        (255, 0, 255),
        1,
    )

fig = px.imshow(image1, width=700)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.show()

Возьмем две точки на изображении, расположенные на платформе. 

Затем, используя известные внутренние и внешние параметры камеры, вычислим их координаты в мировой системе координат и определим расстояние между ними.

In [84]:
image2 = image1.copy()

x1, y1 = 490, 450
x2, y2 = 615, 240

cv2.circle(image2, (x1, y1), 6, (255, 0, 0), -1)
cv2.circle(image2, (x2, y2), 6, (255, 0, 0), -1)

fig = px.imshow(image2, height=500)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.show()

#### **Задача 4**

Используя формулу проекции точки на изображении 
в луч, проходящий через оптический центр камеры:
$$
    \begin{bmatrix}
        u \\ v \\ w \\ 1
    \end{bmatrix}
     = \mathbf{P}^+ \cdot
    \begin{bmatrix}
        x \\ y \\ 1
    \end{bmatrix}  + 
    t \cdot 
    \begin{bmatrix}
        \mathbf{c} \\ 1
    \end{bmatrix}
$$
реализуйте функцию `from_image_coordinates_to_world(x, y, c, P)`, которая 
по координатам точки на изображении $(x, y)$,
оптическому центру камеры $\mathbf{c}$ 
и матрице камеры $\mathbf{P}$ вычисляет координаты точки в мировой системе координат, 
расположенной на плоскости $v=0$ (плоскости платформы).

```python
    # file: homeworks/homework_07.py
    
    def from_image_coordinates_to_world(x, y, c, P):
        ...
        return point3D
```

Проверка функции `from_image_coordinates_to_world`:

In [ ]:
def from_image_coordinates_to_world(x, y, c, P):
    P_plus = P.T @ np.linalg.inv(P @ P.T)
    point3D = P_plus @ np.array([x, y, 1])
    t = -point3D[1] / c[1]
    point3D = point3D + t * np.hstack([c, 1])
    point3D = point3D[:3] / point3D[3]

    return point3D


point3D_1 = from_image_coordinates_to_world(x1, y1, c, P)
point3D_2 = from_image_coordinates_to_world(x2, y2, c, P)

distance = np.linalg.norm(point3D_1 - point3D_2)

cv2.line(image2, (x1, y1), (x2, y2), (0, 255, 0), 3)

fig = px.imshow(
    image2, height=500, title=f"Расстояние между точками равно {distance:.2f} метров."
)
fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False)
fig.show()

### **Домашнее задание 7**

#### **Теоретическая часть**

- Однородные координаты в 2D и 3D

- Замена системы координат с помощью матрицы вращения и вектора сдвига
 (формулы замены в декартовых и однородных координатах)

- Параметризация матрицы вращения с помощью вектора вращения. Формулы Родрига

- Модель pinhole-камеры. Внутренние и внешние параметры камеры

- Матрица камеры. Проекция 3D точек на изображение с помощью матрицы камеры и обратная проекция точек изображения в 3D пространство

- Модель радиальной и тангенциальной дисторсии камеры

##### **Практическая часть**
Реализуйте в файле `homeworks/homework_07.py` функции из задач 1-4 и проверьте их работоспособность.